# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @ids, fields, and columns
print("Available Record Sets (by @id):")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']} ({rs['description'] if 'description' in rs else ''})")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']}: {field['name']}")
    if 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    - {col['@id']}: {col['name']}")
    print()
# Optionally, explore the first record set in more detail
if record_sets:
    example_rs_id = record_sets[0]['@id']
    print(f"Example records from {example_rs_id}:")
    for record in dataset.records(record_set=example_rs_id):
        print(record)
        break  # Print only one sample record

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract from all available record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    # Only create DataFrame if there are records
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns for record set {rs_id}:", dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())
    else:
        print(f"No records found in record set {rs_id}")

# Select one record set for analysis below. You can change this index if desired.
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Automatically find a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # Try to convert columns to numeric silently
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for EDA.")
        threshold = df[numeric_field].quantile(0.50)  # use median as an example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field, e.g. if a column looks like 'description' or 'accession'.
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field to group by in this record set.")
    else:
        print("No numeric fields detected for EDA in this record set.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {selected_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        # Plot top 10 groups by median value
        plt.figure(figsize=(10, 5))
        top_groups = df.groupby(group_field)[numeric_field].median().sort_values(ascending=False).head(10)
        sns.barplot(x=top_groups.index, y=top_groups.values)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Top 10 {group_field} by median {numeric_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.